# M2 상품군·가격 구매이력 표현 — 체크포인트 분해진단

기존 seed 42 역사적 개발실험 체크포인트만 재사용합니다. 재학습·epoch 선택·최종 test 평가는 하지 않습니다.

- ID-only
- ID+상품군 이력(상품군 축 완전 활성화)
- ID+가격 이력(가격 축 완전 활성화)
- 학습된 결합
- 0.5/0.5 균등 결합
- 사용자 CLV 조건 셔플


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
![ -d /content/clv-m2-lightgcn-runner ] || git clone -q --branch feat/m2-joint-nv-lightgcn https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git fetch -q origin feat/m2-joint-nv-lightgcn
!git checkout -q feat/m2-joint-nv-lightgcn
!git pull -q --ff-only origin feat/m2-joint-nv-lightgcn
print('진단 코드 준비 완료')

In [ ]:
import json
import torch
from lightgcn_clv_conditioned_category_price_history_diagnostic import (
    configure_decomposition_diagnostic,
    preflight_summary,
    run_conditioned_history_decomposition,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_decomposition_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_clv_conditioned_category_price_history_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['final_test_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
report = run_conditioned_history_decomposition(cfg)

In [ ]:
from IPython.display import display

print('1) ID / 상품군 / 가격 / 결합 방식별 전체·CLV 구간 성과')
display(report['view_metrics'])
print('2) ID-only 대비 핵심 변화')
display(report['core'])
print('3) 실제 적용된 결합 비율')
display(report['gate_summary'])
print('결과 파일:', report['paths'])